# Lab 1 vs Lab 2 — Comparación de modelos

<a href="https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab2/lab1vslab2_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook entrena la ANN ganadora de Lab 1 y la evalúa sobre **exactamente las mismas canciones** que forman el conjunto de test de Lab 2 Task 1 y Task 2.

**Partición compartida:** se replica el procedimiento de los notebooks de Lab 2: recorrer las carpetas de géneros en orden alfabético, recorrer los nombres de imagen ordenados, y aplicar dos divisiones estratificadas con `random_state=42` (64 % train, 16 % validation, 20 % test).

**Emparejamiento:** se extrae el identificador de canción del nombre de archivo del CSV y de cada imagen. El notebook exige correspondencia uno a uno y etiquetas de género coherentes. Si faltan identificadores o hay duplicados, se detiene antes de entrenar.

Así, la comparación de test queda emparejada por canción. Las entradas siguen siendo distintas: Lab 1 usa características tabulares y Lab 2 usa imágenes.

## 1. Montar Google Drive y configurar rutas

El CSV de características y la carpeta de imágenes deben corresponder al mismo dataset GTZAN.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CSV_PATH = Path("/content/drive/MyDrive/MSLS/Data/features_30_sec.csv")
DATA_DIR = Path("/content/drive/MyDrive/MLSM/Data/images_original")
OUTPUT_DIR = Path("/content/drive/MyDrive/MSLS/Results/lab1vslab2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encuentra el CSV: {CSV_PATH}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"No se encuentra la carpeta de imágenes: {DATA_DIR}")

print("CSV:", CSV_PATH)
print("Imágenes:", DATA_DIR)
print("Resultados:", OUTPUT_DIR)

## 2. Importaciones y reproducibilidad

Fijamos la semilla utilizada en Lab 2. Esto hace reproducible la partición siempre que el CSV, el orden de sus filas y las etiquetas no cambien.

In [ ]:
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 3. Cargar y preparar el CSV

La columna `label` contiene el género. El modelo recibe únicamente columnas numéricas. Las columnas de texto, como `filename`, no se utilizan como características.

In [ ]:
# Cargar características tabulares.
df = pd.read_csv(CSV_PATH)
print("Dimensiones iniciales del CSV:", df.shape)

required_columns = {"label", "filename"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"Faltan columnas necesarias en el CSV: {sorted(missing_columns)}. "
        "Se necesita 'filename' para emparejar cada canción con su imagen."
    )

df = df.dropna(subset=["label", "filename"]).copy()
df["label"] = df["label"].astype(str).str.strip()

# Normalización del identificador:
# basename + eliminar extensión + conservar solo letras y números.
# Así, por ejemplo, blues.00001.wav y blues00001.png comparten ID blues00001.
import re
def canonical_song_id(filename):
    stem = Path(str(filename).replace("\\", "/")).name
    stem = Path(stem).stem
    return re.sub(r"[^a-zA-Z0-9]", "", stem).lower()

df["song_id"] = df["filename"].map(canonical_song_id)

if (df["song_id"] == "").any():
    raise ValueError("Hay filenames del CSV que generan un identificador vacío.")
if df["song_id"].duplicated().any():
    duplicates = df.loc[df["song_id"].duplicated(False), ["filename", "song_id"]]
    raise ValueError(
        "Hay identificadores duplicados en el CSV. Revisa estas filas:\\n"
        + duplicates.head(20).to_string(index=False)
    )

# Construir el manifiesto siguiendo EXACTAMENTE el orden de recorrido
# usado en Lab 2 Task 1 y Task 2: géneros ordenados y archivos ordenados.
class_names = sorted(
    folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()
)
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp")
image_rows = []

for label_id, genre in enumerate(class_names):
    genre_folder = DATA_DIR / genre
    for image_path in sorted(genre_folder.iterdir(), key=lambda p: p.name):
        if image_path.is_file() and image_path.suffix.lower() in valid_extensions:
            image_rows.append({
                "image_path": str(image_path),
                "song_id": canonical_song_id(image_path.name),
                "genre": genre,
                "image_label": label_id
            })

image_manifest = pd.DataFrame(image_rows)
if image_manifest.empty:
    raise ValueError(f"No se encontraron imágenes en {DATA_DIR}.")
if (image_manifest["song_id"] == "").any():
    raise ValueError("Hay imágenes que generan un identificador vacío.")
if image_manifest["song_id"].duplicated().any():
    duplicates = image_manifest.loc[
        image_manifest["song_id"].duplicated(False), ["image_path", "song_id"]
    ]
    raise ValueError(
        "Hay identificadores de imagen duplicados:\\n"
        + duplicates.head(20).to_string(index=False)
    )

# Comprobar correspondencia exacta entre los dos recursos.
csv_ids = set(df["song_id"])
image_ids = set(image_manifest["song_id"])
missing_images = sorted(csv_ids - image_ids)
missing_csv = sorted(image_ids - csv_ids)

if missing_images or missing_csv:
    raise ValueError(
        "CSV e imágenes no contienen exactamente las mismas canciones.\\n"
        f"IDs del CSV sin imagen: {len(missing_images)}; "
        f"ejemplos: {missing_images[:10]}\\n"
        f"IDs de imagen sin CSV: {len(missing_csv)}; "
        f"ejemplos: {missing_csv[:10]}\\n"
        "No se continúa para evitar comparar particiones distintas."
    )

# Verificar que el género del CSV coincide con la carpeta de la imagen.
csv_genre_by_id = df.set_index("song_id")["label"]
image_genre_by_id = image_manifest.set_index("song_id")["genre"]
genre_mismatches = [
    song_id for song_id in csv_ids
    if csv_genre_by_id.loc[song_id] != image_genre_by_id.loc[song_id]
]
if genre_mismatches:
    details = [
        (sid, csv_genre_by_id.loc[sid], image_genre_by_id.loc[sid])
        for sid in genre_mismatches[:10]
    ]
    raise ValueError(
        "Hay diferencias de género entre CSV e imágenes. "
        f"Ejemplos (ID, CSV, imagen): {details}"
    )

# X se indexa por ID para poder recuperar las filas en el orden exacto
# de las particiones construidas a partir de las rutas de imagen.
feature_df = df.drop(columns=["label", "filename", "song_id"], errors="ignore")
feature_df = feature_df.select_dtypes(include=[np.number]).copy()
if feature_df.empty:
    raise ValueError("No se encontraron características numéricas en el CSV.")

X_by_id = feature_df.copy()
X_by_id.index = df["song_id"].values

print("Géneros:", class_names)
print("Canciones en CSV:", len(df))
print("Imágenes encontradas:", len(image_manifest))
print("Correspondencia 1:1 verificada:", len(csv_ids) == len(image_ids))
print("Características numéricas:", X_by_id.shape[1])
display(image_manifest.head())


## 4. Partición 64/16/20 — igual procedimiento que Lab 2

Primero se reserva el 20 % para test. Después, el 20 % del 80 % restante se reserva para validación. Ambas divisiones son estratificadas y usan `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Importante: las divisiones se hacen sobre las mismas rutas y etiquetas,
# en el mismo orden, que en los notebooks de Lab 2.
image_paths = image_manifest["image_path"].to_numpy()
image_labels = image_manifest["image_label"].to_numpy()
image_song_ids = image_manifest["song_id"].to_numpy()

dev_paths, test_paths, dev_labels, test_labels, dev_ids, test_ids = train_test_split(
    image_paths,
    image_labels,
    image_song_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=image_labels
)

train_paths, val_paths, train_labels, val_labels, train_ids, val_ids = train_test_split(
    dev_paths,
    dev_labels,
    dev_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=dev_labels
)

# Recuperar las filas del CSV en el orden exacto de cada split de imágenes.
X_train = X_by_id.loc[train_ids].copy()
X_val = X_by_id.loc[val_ids].copy()
X_test = X_by_id.loc[test_ids].copy()

# Usar las etiquetas del manifiesto de imágenes, ya verificadas contra el CSV.
y_train = train_labels.astype(int)
y_val = val_labels.astype(int)
y_test = test_labels.astype(int)

# Comprobaciones explícitas: los IDs son disjuntos y cubren todo el dataset.
assert set(train_ids).isdisjoint(set(val_ids))
assert set(train_ids).isdisjoint(set(test_ids))
assert set(val_ids).isdisjoint(set(test_ids))
assert len(train_ids) + len(val_ids) + len(test_ids) == len(image_manifest)

split_summary = pd.DataFrame({
    "Conjunto": ["Train", "Validation", "Test", "Total"],
    "Canciones": [len(train_ids), len(val_ids), len(test_ids), len(image_song_ids)]
})
split_summary["Porcentaje"] = 100 * split_summary["Canciones"] / len(image_song_ids)
display(split_summary)

print("Partición idéntica al algoritmo de Lab 2:")
print("Train IDs:", len(train_ids))
print("Validation IDs:", len(val_ids))
print("Test IDs:", len(test_ids))
print("Ejemplos de IDs de test:", list(test_ids[:10]))

# Guardar el manifiesto de partición común para trazabilidad.
split_ids = pd.DataFrame({
    "song_id": np.concatenate([train_ids, val_ids, test_ids]),
    "split": (
        ["train"] * len(train_ids)
        + ["validation"] * len(val_ids)
        + ["test"] * len(test_ids)
    )
})
split_ids.to_csv(OUTPUT_DIR / "shared_song_split.csv", index=False)


## 5. Definir la ANN ganadora de Lab 1

Se conservan los hiperparámetros seleccionados en Lab 1:
- Capas ocultas: (128, 64)
- `alpha=0.001`
- `learning_rate_init=0.001`
- Optimizador: Adam
- `max_iter=10000`
- Semilla: 42

El preprocesamiento se ajusta dentro del pipeline. Así, la mediana, el escalado y la selección de características se calculan únicamente con el conjunto de entrenamiento y no filtran información de validación/test.

In [ ]:
ann_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("variance", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k="all")),
    ("model", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        solver="adam",
        learning_rate_init=0.001,
        alpha=0.001,
        max_iter=10000,
        random_state=SEED,
        early_stopping=False,
        n_iter_no_change=15,
        tol=1e-3
    ))
])

display(ann_pipeline)

## 6. Entrenar la ANN

El modelo se entrena solo con train. Validation se reserva para observar generalización; no se utiliza para ajustar pesos porque el MLP conserva `early_stopping=False`, como en la configuración ganadora original.

In [ ]:
ann_pipeline.fit(X_train, y_train)
mlp = ann_pipeline.named_steps["model"]

print("Entrenamiento completado.")
print("Iteraciones realizadas:", mlp.n_iter_)
print("Loss final:", mlp.loss_)
print("Convergencia indicada por sklearn:", mlp.n_iter_ < mlp.max_iter)

## 7. Curva de pérdida del entrenamiento

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(mlp.loss_curve_, label="Train loss")
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("Lab 1 — Pérdida durante el entrenamiento")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Métricas de train, validation y test

La métrica principal para la comparación con Lab 2 es la accuracy de test. Train y validation ayudan a diagnosticar sobreajuste.

In [ ]:
def evaluate_split(name, features, labels):
    pred = ann_pipeline.predict(features)
    return {
        "Split": name,
        "N": len(labels),
        "Accuracy": accuracy_score(labels, pred),
        "Precision macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[0],
        "Recall macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[1],
        "F1 macro": precision_recall_fscore_support(
            labels, pred, average="macro", zero_division=0
        )[2],
    }

ann_metrics_df = pd.DataFrame([
    evaluate_split("Train", X_train, y_train),
    evaluate_split("Validation", X_val, y_val),
    evaluate_split("Test", X_test, y_test),
])
display(ann_metrics_df.round(4))

ann_test_accuracy = float(ann_metrics_df.loc[
    ann_metrics_df["Split"] == "Test", "Accuracy"
].iloc[0])

## 9. Informe de clasificación y matrices de confusión

La diagonal de la matriz representa las predicciones correctas. La matriz normalizada permite comparar el recall de cada género independientemente de cuántas muestras tenga.

In [ ]:
y_test_pred = ann_pipeline.predict(X_test)

print(classification_report(
    y_test, y_test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
    digits=4
))

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False
)
ax.set_title("Lab 1 — Matriz de confusión (test)")
plt.tight_layout()
plt.show()

cm_norm = confusion_matrix(
    y_test, y_test_pred,
    labels=np.arange(len(class_names)),
    normalize="true"
)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_xlabel("Género predicho")
ax.set_ylabel("Género real")
ax.set_title("Lab 1 — Matriz de confusión normalizada")
plt.tight_layout()
plt.show()

## 10. Guardar resultados de Lab 1

Se guardan las métricas, predicciones, nombres de clase y curva de pérdida en Drive para poder consultar los resultados después.

In [ ]:
lab1_result = {

    "model": "MLPClassifier",

    "seed": SEED,

    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},

    "n_samples": int(len(X_by_id)),

    "n_train": int(len(X_train)),

    "n_validation": int(len(X_val)),

    "n_test": int(len(X_test)),

    "class_names": [str(x) for x in class_names],

    "test_accuracy": ann_test_accuracy,

    "metrics": ann_metrics_df.to_dict(orient="records"),

    "hyperparameters": {

        "hidden_layer_sizes": [128, 64],

        "alpha": 0.001,

        "learning_rate_init": 0.001,

        "solver": "adam",

        "max_iter": 10000,

        "early_stopping": False,

        "n_iter_no_change": 15,

        "tol": 0.001

    }

}



with open(OUTPUT_DIR / "lab1_metrics.json", "w") as f:

    json.dump(lab1_result, f, indent=2)



np.savez(

    OUTPUT_DIR / "lab1_test_predictions.npz",

    y_true=y_test,

    y_pred=y_test_pred,

    class_names=class_names,
    test_song_ids=test_ids,
    test_image_paths=test_paths

)



pd.DataFrame({

    "iteration": np.arange(1, len(mlp.loss_curve_) + 1),

    "loss": mlp.loss_curve_

}).to_csv(OUTPUT_DIR / "lab1_loss_curve.csv", index=False)



ann_metrics_df.to_csv(OUTPUT_DIR / "lab1_split_metrics.csv", index=False)



print("Resultados guardados en:", OUTPUT_DIR)

## 11. Comparación con Lab 2 Task 1 y Task 2

Introduce las accuracies de test que aparecen al final de los notebooks de Lab 2. No se leen automáticamente porque las versiones actuales de esos notebooks no exportan una métrica final común a un archivo compartido. Se dejan como `None` para evitar utilizar accidentalmente resultados antiguos o inventados.

In [ ]:
# Sustituye los valores None por la accuracy de test de las ejecuciones
# de Lab 2 que quieras comparar. Ejemplo: 0.745
TASK1_TEST_ACCURACY = None  # CNN desde cero
TASK2_TEST_ACCURACY = None  # VGG16 transfer learning

comparison = pd.DataFrame([
    {"Modelo": "Lab 1 — ANN", "Test accuracy": ann_test_accuracy},
    {"Modelo": "Lab 2 Task 1 — CNN", "Test accuracy": TASK1_TEST_ACCURACY},
    {"Modelo": "Lab 2 Task 2 — VGG16", "Test accuracy": TASK2_TEST_ACCURACY},
])

display(comparison)

available = comparison.dropna(subset=["Test accuracy"])
if len(available) > 0:
    plt.figure(figsize=(9, 5))
    bars = plt.bar(available["Modelo"], available["Test accuracy"])
    plt.ylim(0, 1)
    plt.ylabel("Accuracy")
    plt.title("Comparación de accuracy en test")
    plt.xticks(rotation=15, ha="right")
    for bar, value in zip(bars, available["Test accuracy"]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.015,
            f"{value:.3f}",
            ha="center"
        )
    plt.tight_layout()
    plt.show()
else:
    print("Introduce al menos una accuracy de Lab 2 para generar la gráfica.")

## 12. Interpretación y límites

- La accuracy de test estima el rendimiento sobre datos no utilizados para ajustar los pesos.
- La precision, recall y F1 macro dan el mismo peso a cada género.
- Una diferencia grande entre train y test puede indicar sobreajuste.
- La MLP recibe características tabulares; la CNN y VGG16 reciben imágenes. No tienen arquitecturas ni parámetros de optimización equivalentes.
- En este notebook, la partición se crea una sola vez a partir del manifiesto de imágenes y sus identificadores se reutilizan para seleccionar las filas del CSV. Así, Lab 1 y Lab 2 usan las mismas canciones en cada split, siempre que las comprobaciones de correspondencia entre archivos terminen correctamente.

**Ruta de salida:** `MyDrive/MSLS/Results/lab1vslab2/`.